In [15]:
import scanpy as sc
import os
import seaborn as sns
import matplotlib.pyplot as plt
import gzip

In [ ]:
# Extracting raw data
!tar -xf raw_data/GSE235063_RAW.tar

In [ ]:
# Moving extracted data to a new folder 'raw_data'
!mv GSM* raw_data/

In [ ]:
# Checking move was successful and creating a features file list
files = os.listdir('raw_data')
ffs = [x for x in files if 'features' in x]

In [ ]:
# Creating a temporary directory to add Gene Expression column to all feature files
!mkdir temp

In [ ]:
# Adding Gene Expression column to feature files
for ff in ffs:
    with gzip.open('raw_data/' + ff, 'rt') as f_in:
        with gzip.open('temp/' + ff, 'wt') as f_out:
            for line in f_in:
                f_out.write(line.strip() + '\tGene Expression\n')

In [ ]:
# Checking column addition was successful
!gzcat temp/GSM7494257_AML16_DX_raw_features.tsv.gz | head

ENSG00000243485	MIR1302-2HG	Gene Expression
ENSG00000237613	FAM138A	Gene Expression
ENSG00000186092	OR4F5	Gene Expression
ENSG00000238009	AL627309.1	Gene Expression
ENSG00000239945	AL627309.3	Gene Expression
ENSG00000239906	AL627309.2	Gene Expression
ENSG00000241599	AL627309.4	Gene Expression
ENSG00000236601	AL732372.1	Gene Expression
ENSG00000284733	OR4F29	Gene Expression
ENSG00000235146	AC114498.1	Gene Expression
gzcat: error writing to output: Broken pipe
gzcat: temp/GSM7494257_AML16_DX_raw_features.tsv.gz: uncompress failed


In [ ]:
# Replcaing raw_data folder with updated feature files
!mv temp/* raw_data/

In [ ]:
# Checking raw sample to ensure scanpy is reading the files correctly
sc.read_10x_mtx('raw_data/', prefix = 'GSM7494257_AML16_DX_raw_')

AnnData object with n_obs × n_vars = 6794880 × 33538
    var: 'gene_ids', 'feature_types'
    layers: None (.X)

In [ ]:
# Creating a raw_adata folder to store adata objects
!mkdir raw_adata

In [ ]:
# Creating adata objects for all raw files that have not been processed
for prefix in set([x.split('_raw')[0] + '_raw_' for x in files if 'processed' not in x and 'Store' not in x]):
    adata = sc.read_10x_mtx('raw_data/', prefix = prefix)
    adata.write_h5ad('raw_adata/' + prefix + '.h5ad')